# 01. Data Ingestion & Preprocessing
ขั้นตอนการนำเข้าข้อมูลหุ้น 20 ตัว และข้อมูลปัจจัยมหภาค พร้อมการจัดการค่าว่าง/ค่าซ้ำ และแปลงชื่อคอลัมน์มาตรฐาน

# ข้อมูลดิบ

###โหลดฟ้อนต์

In [1]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import urllib.request

# 1. ดาวน์โหลดฟอนต์ภาษาไทย TH Sarabun New เพื่อใช้แสดงผลบนกราฟ
font_url = 'https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf'
font_path = './Sarabun-Regular.ttf'
urllib.request.urlretrieve(font_url, font_path)

# 2. ลงทะเบียนฟอนต์กับระบบ Matplotlib
fm.fontManager.addfont(font_path)

# 3. กำหนดค่าเริ่มต้นให้ Matplotlib ดึงฟอนต์ Sarabun ไปใช้งานเป็นหลัก
plt.rcParams['font.family'] = 'Sarabun'
plt.rcParams['axes.unicode_minus'] = False

###เชื่อมต่อไดรฟ์

In [2]:
# ตรวจสอบสภาพแวดล้อม (Local vs Google Colab)
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    print('✓ เชื่อมต่อ Google Drive สำเร็จ (Google Colab)')
except (ImportError, ModuleNotFoundError):
    print('✓ กำลังรันบน Local Environment (ข้ามการเชื่อมต่อ Google Drive)')


✓ กำลังรันบน Local Environment (ข้ามการเชื่อมต่อ Google Drive)


# FUNCTION

## check duplicate

In [3]:
import pandas as pd
from IPython.display import display

def check_duplicates(df):
    """
    ตรวจสอบหาแถวที่ซ้ำซ้อน (Duplicate Rows) ใน DataFrame
    """
    duplicate_rows = df[df.duplicated()]
    total_duplicates = len(duplicate_rows)

    if total_duplicates > 0:
        print(f"⚠ พบแถวซ้ำทั้งหมด: {total_duplicates} แถว")
        print("ตัวอย่างข้อมูลที่ซ้ำซ้อน:")
        display(duplicate_rows.head(10))
    else:
        print("✓ ไม่พบข้อมูลซ้ำซ้อน (No Duplicates)")

    return duplicate_rows

## check null

In [4]:
import pandas as pd
from IPython.display import display

def check_and_show_nulls(df):
    """
    ตรวจสอบหาค่าว่าง (Null/NaN) ใน DataFrame และพิมพ์สรุปจำนวนแถวพร้อมแสดงข้อมูล
    """
    null_counts = df.isnull().sum()
    cols_with_null = null_counts[null_counts > 0]

    if cols_with_null.empty:
        print("✓ ไม่พบค่าว่างในข้อมูลนี้")
        return None

    null_rows = df[df.isnull().any(axis=1)]

    print(f"⚠ พบค่าว่างในคอลัมน์:")
    for col, count in cols_with_null.items():
        print(f"  - '{col}': {count} แถว")

    print(f"\nรวมแถวที่มีค่าว่างทั้งหมด: {len(null_rows)} แถว")
    display(null_rows.head(10))

    return null_rows

## delete null

In [5]:
import pandas as pd

def drop_null_rows(df):
    """
    ลบแถวที่มีค่าว่าง (Null/NaN) ใน DataFrame และพิมพ์สรุปจำนวนที่ถูกลบ
    """
    initial_count = len(df)
    df_clean = df.dropna()
    deleted_count = initial_count - len(df_clean)

    if deleted_count > 0:
        print(f"✓ ลบค่าว่างสำเร็จ: ทั้งหมด {deleted_count} แถว")
    else:
        print("✓ ไม่พบค่าว่างในข้อมูลนี้")

    return df_clean

## drop column

In [6]:
import pandas as pd

def drop_columns(df, columns_to_drop):
    """
    ลบคอลัมน์ที่ระบุออกจาก DataFrame อย่างปลอดภัย
    """
    # เลือกเฉพาะคอลัมน์ที่มีอยู่ใน DataFrame จริงๆ เพื่อเลี่ยง Error
    existing_cols = [col for col in columns_to_drop if col in df.columns]

    if existing_cols:
        df_clean = df.drop(columns=existing_cols)
        print(f"✓ ลบคอลัมน์สำเร็จ: {existing_cols}")
    else:
        print("✓ ไม่พบคอลัมน์ที่ระบุใน DataFrame (ไม่มีการลบ)")
        df_clean = df.copy()

    return df_clean

## Bfill and Ffill

In [7]:
import pandas as pd

def fill_null_values(df):
    """
    จัดการเติมค่าว่าง (Null/NaN) ด้วยวิธี Forward Fill (ffill) ตามด้วย Backward Fill (bfill)
    """
    initial_nulls = df.isnull().sum().sum()

    if initial_nulls == 0:
        print("✓ ไม่พบค่าว่างในข้อมูลนี้")
        return df.copy()

    df_filled = df.ffill().bfill()
    remaining_nulls = df_filled.isnull().sum().sum()

    print(f"✓ จัดการค่าว่างสำเร็จ (เริ่มต้น {initial_nulls} ค่า -> คงเหลือ {remaining_nulls} ค่า)")
    return df_filled

# ค้นหา DataFrame ทั้งหมดที่มีค่าว่างและนำมาจัดการเติมข้อมูล
all_dfs = {name: val for name, val in globals().items() if isinstance(val, pd.DataFrame) and not name.startswith('_')}

print("=== กำลังเริ่มทำการจัดการค่าว่างด้วยวิธี Forward / Backward Fill ===\n")
for name, df in sorted(all_dfs.items()):
    if name == 'df':
        continue

    null_count = df.isnull().sum().sum()
    if null_count > 0:
        print(f"[{name}] พบค่าว่าง {null_count} ตำแหน่ง")
        globals()[name] = fill_null_values(df)
        print()

=== กำลังเริ่มทำการจัดการค่าว่างด้วยวิธี Forward / Backward Fill ===



## scatter graph

In [8]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import pandas as pd
import numpy as np
from IPython.display import display

# ตั้งค่าสไตล์และการแสดงผลเบื้องต้น พร้อมกำหนดฟอนต์สำหรับกรณีทั่วไป
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'Sarabun'
plt.rcParams['axes.unicode_minus'] = False

def analyze_correlation_generic(df_macro, df_assets,
                               date_col='Date',
                               macro_col='Close',
                               asset_col='Close',
                               group_col='Stock',
                               macro_name='USD_THB',
                               asset_type='หุ้นส่งออก',
                               color='#2ca02c'):
    """
    ฟังก์ชันวิเคราะห์และพล็อต Regression ระหว่างปัจจัยมหภาคกับกลุ่มสินทรัพย์ โดยระบุฟอนต์ภาษาไทยตรงไปยังออบเจกต์ของกราฟโดยตรง
    และเพิ่มพารามิเตอร์ color เพื่อกำหนดสีของจุดใน Scatter plot (Default: สีเขียว '#2ca02c')
    """
    # ดึงออบเจกต์ฟอนต์ Sarabun โดยตรงเพื่อความชัวร์
    thai_font = fm.FontProperties(fname='./Sarabun-Regular.ttf')

    # 1. เตรียมข้อมูลปัจจัยมหภาค
    macro_renamed = f"{macro_name}_{macro_col}"
    df_macro_clean = df_macro[[date_col, macro_col]].rename(columns={macro_col: macro_renamed})
    df_macro_clean[date_col] = pd.to_datetime(df_macro_clean[date_col])

    # รายชื่อกลุ่มสินทรัพย์ทั้งหมดที่ต้องการวิเคราะห์
    unique_groups = df_assets[group_col].unique()
    num_groups = len(unique_groups)

    # คำนวณจำนวนแถวสำหรับ Subplots (แสดง 2 คอลัมน์)
    num_cols = 2
    num_rows = int(np.ceil(num_groups / num_cols))

    # เคลียร์ข้อมูลสำหรับการบันทึกค่า Correlation
    saved_correlations = []

    # 2. สร้าง Grid สำหรับแสดงผลกราฟ
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(16, 4.8 * num_rows))
    if num_groups == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()  # แปลงเป็น 1D array

    for idx, group_name in enumerate(unique_groups):
        if idx >= len(axes):
            break

        # กรองข้อมูลเฉพาะกลุ่มย่อยรายตัว
        asset_renamed = f"Asset_{asset_col}"
        df_sub = df_assets[df_assets[group_col] == group_name][[date_col, asset_col]].rename(columns={asset_col: asset_renamed})
        df_sub[date_col] = pd.to_datetime(df_sub[date_col])

        # รวมข้อมูลปัจจัยมหภาคกับข้อมูลสินทรัพย์รายย่อย
        df_merged = pd.merge(df_macro_clean, df_sub, on=date_col, how='inner').sort_values(date_col)

        if df_merged.empty or len(df_merged) < 2:
            continue

        # คำนวณค่าสหสัมพันธ์ (Correlation)
        correlation = df_merged[macro_renamed].corr(df_merged[asset_renamed])

        # บันทึกค่าลงในรายการ
        saved_correlations.append({
            'Group': group_name,
            'ประเภทสินทรัพย์': asset_type,
            'Macro Factor': macro_name,
            'Correlation': correlation
        })

        # วาด Regression Plot โดยใช้ตัวแปรสีที่ส่งเข้ามา
        sns.regplot(
            data=df_merged,
            x=macro_renamed,
            y=asset_renamed,
            ax=axes[idx],
            scatter_kws={'alpha': 0.25, 'color': color, 's': 10},
            line_kws={'color': '#d62728', 'linewidth': 1.5}
        )

        # ตั้งชื่อและป้ายกำกับด้วยฟอนต์ภาษาไทยอย่างเฉพาะเจาะจง
        axes[idx].set_title(f'{group_name} (Corr = {correlation:.3f})', fontproperties=thai_font, fontsize=12, fontweight='bold')
        axes[idx].set_xlabel(f'{macro_name} ({macro_col})', fontproperties=thai_font, fontsize=11)
        axes[idx].set_ylabel(f'{group_name} ({asset_col})', fontproperties=thai_font, fontsize=11)

    # ซ่อนแกนที่เหลือ (กรณีจำนวนกลุ่มเป็นเลขคี่)
    for j in range(idx + 1, len(axes)):
        fig.delaxes(axes[j])

    # ตั้งค่าหัวข้อใหญ่ด้วยฟอนต์ภาษาไทย
    fig.suptitle(f'Correlation between {macro_name} and {asset_type}', fontproperties=thai_font, fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    # --- แสดงตารางสรุป ---
    df_summary = pd.DataFrame(saved_correlations).drop_duplicates(subset=['Group', 'Macro Factor']).reset_index(drop=True)
    df_sorted = df_summary.sort_values(by='Correlation').reset_index(drop=True)

    print(f"\n=== [ตารางสรุป] ค่า Correlation ระหว่าง {macro_name} และ {asset_type} (เรียงจากน้อยไปมาก) ===")
    display(df_sorted)

    return df_sorted

## scale corr

In [9]:
def scale_correlation(corr_value, factor=10):
    """
    ฟังก์ชันสำหรับคูณสเกลค่า Correlation เพื่อขยายผลลัพธ์
    """
    if corr_value is None or pd.isna(corr_value):
        return corr_value
    return corr_value * factor


# IMPORT DATA

In [10]:
import os
import glob
from pathlib import Path

# ระบบตรวจสอบ Path อัตโนมัติ: รองรับทั้ง Local Machine และ Google Colab
local_candidates = [
    Path('../data/01_raw'),
    Path('data/01_raw'),
    Path('./data/01_raw')
]
colab_base = Path('/content/drive/MyDrive/Colab Notebooks/DSDN')

raw_base = None
for cand in local_candidates:
    if cand.exists():
        raw_base = cand.resolve()
        print(f'✓ ตรวจพบสภาพแวดล้อม: Local Machine (โฟลเดอร์: {raw_base})')
        break

if raw_base is None and colab_base.exists():
    raw_base = colab_base
    print('✓ ตรวจพบสภาพแวดล้อม: Google Colab (เชื่อมต่อ Google Drive)')

if raw_base is None:
    raise FileNotFoundError('ไม่พบโฟลเดอร์ data/01_raw กรุณาตรวจสอบตำแหน่งไฟล์')

def get_path(folder_names):
    if isinstance(folder_names, str):
        folder_names = [folder_names]
    for fn in folder_names:
        p = raw_base / fn
        if p.exists():
            return str(p / '*.csv')
    return str(raw_base / folder_names[0] / '*.csv')

path_import       = get_path('หุ้นนำเข้า')
path_export       = get_path('หุ้นส่งออก')
path_interest     = get_path('ดอกเบี้ย')
path_fx           = get_path('อัตราแลกเปลี่ยน')
path_th_interest  = get_path('ดอกเบี้ยอ้างอิงแห่งประเทศไทย')
path_oil          = get_path('ราคาน้ำมัน')
path_th_bond      = get_path('พันธบัตรไทย')
path_indices      = get_path(['ข้อมูลดัชนี', 'ดัชนี'])

files_import      = glob.glob(path_import)
files_export      = glob.glob(path_export)
files_interest    = glob.glob(path_interest)
files_fx          = glob.glob(path_fx)
files_th_interest = glob.glob(path_th_interest)
files_oil         = glob.glob(path_oil)
files_th_bond     = glob.glob(path_th_bond)
files_indices     = glob.glob(path_indices)

print("จำนวนไฟล์ในโฟลเดอร์หุ้นนำเข้า:", len(files_import))
print("จำนวนไฟล์ในโฟลเดอร์หุ้นส่งออก:", len(files_export))
print("จำนวนไฟล์ในโฟลเดอร์ดอกเบี้ย:", len(files_interest))
print("จำนวนไฟล์ในโฟลเดอร์อัตราแลกเปลี่ยน:", len(files_fx))
print("จำนวนไฟล์ในโฟลเดอร์ดอกเบี้ยอ้างอิง:", len(files_th_interest))
print("จำนวนไฟล์ในโฟลเดอร์ราคาน้ำมัน:", len(files_oil))
print("จำนวนไฟล์ในโฟลเดอร์พันธบัตร:", len(files_th_bond))
print("จำนวนไฟล์ในโฟลเดอร์ข้อมูลดัชนี:", len(files_indices))


✓ ตรวจพบสภาพแวดล้อม: Local Machine (โฟลเดอร์: /Users/pathcharapisit/Desktop/newdsde/data/01_raw)
จำนวนไฟล์ในโฟลเดอร์หุ้นนำเข้า: 10
จำนวนไฟล์ในโฟลเดอร์หุ้นส่งออก: 10
จำนวนไฟล์ในโฟลเดอร์ดอกเบี้ย: 2
จำนวนไฟล์ในโฟลเดอร์อัตราแลกเปลี่ยน: 4
จำนวนไฟล์ในโฟลเดอร์ดอกเบี้ยอ้างอิง: 1
จำนวนไฟล์ในโฟลเดอร์ราคาน้ำมัน: 2
จำนวนไฟล์ในโฟลเดอร์พันธบัตร: 2
จำนวนไฟล์ในโฟลเดอร์ข้อมูลดัชนี: 7


In [11]:
import os
import pandas as pd

# ฟังก์ชันเสริมสำหรับโหลดไฟล์ CSV และตั้งชื่อตัวแปรตามชื่อไฟล์แบบอัตโนมัติ
def load_data_from_paths(file_list, prefix="df_"):
    loaded_dfs = {}
    for path in file_list:
        filename = os.path.splitext(os.path.basename(path))[0]
        clean_name = filename.replace("ราคาย้อนหลัง หุ้น ", "") \
                             .replace("ราคาย้อนหลังของ สัญญาซื้อขายล่วงหน้าน้ำมันดิบ ", "") \
                             .replace("ราคาย้อนหลังของ สัญญาซื้อขายล่วงหน้าน้ำมัน", "") \
                             .replace("ข้อมูลผลตอบแทนย้อนหลังของพันธบัตร ", "") \
                             .replace("อัตราแลกเปลี่ยนย้อนหลัง ", "") \
                             .replace("ข้อมูลย้อนหลังของ ดัชนี ", "") \
                             .replace("ข้อมูลย้อนหลังของ ", "") \
                             .replace("FM_RT_013_formatted", "TH_RATE")

        # จัดการแปลงชื่อเฉพาะกลุ่มอื่นๆ
        if "สหรัฐอเมริกา 10 ปี" in clean_name or "สหรัฐ 10 ปี" in clean_name:
            clean_name = "US_10Y"
        elif "สหรัฐอเมริกา 30 ปี" in clean_name or "สหรัฐ 30 ปี" in clean_name:
            clean_name = "US_30Y"
        elif "ไทย 10 ปี" in clean_name:
            clean_name = "TH_10Y"
        elif "ไทย 20 ปี" in clean_name:
            clean_name = "TH_20Y"
        elif "Baltic Dry Index" in clean_name:
            clean_name = "BDI"
        elif "CBOE Volatility Index" in clean_name:
            clean_name = "VIX"
        elif "Philadelphia Semiconductor Index" in clean_name:
            clean_name = "SOX"
        elif "S&P 500" in clean_name:
            clean_name = "SP500"
        elif "SET_50" in clean_name or "SET 50" in clean_name:
            clean_name = "SET50"
        elif "US Dollar Index" in clean_name:
            clean_name = "DXY"

        clean_name = clean_name.replace(" ", "_")
        var_name = f"{prefix}{clean_name}"

        try:
            df_temp = pd.read_csv(path)
            globals()[var_name] = df_temp
            loaded_dfs[var_name] = df_temp
            print(f"✓ โหลด {var_name} สำเร็จ ({df_temp.shape[0]} แถว x {df_temp.shape[1]} คอลัมน์)")
        except Exception as e:
            print(f"✗ ไม่สามารถโหลด {var_name} จาก {path} ได้: {e}")
    return loaded_dfs

print("=== เริ่มต้นการโหลดข้อมูลทั้งหมดเข้าสู่ระบบ ===\n")

print("--- [1/7] โหลดข้อมูลหุ้นนำเข้า ---")
load_data_from_paths(files_import)

print("\n--- [2/7] โหลดข้อมูลหุ้นส่งออก ---")
load_data_from_paths(files_export)

print("\n--- [3/7] โหลดข้อมูลอัตราแลกเปลี่ยน ---")
load_data_from_paths(files_fx)

print("\n--- [4/7] โหลดข้อมูลดอกเบี้ยสหรัฐฯ ---")
load_data_from_paths(files_interest)

print("\n--- [5/7] โหลดข้อมูลพันธบัตรไทย ---")
load_data_from_paths(files_th_bond)

print("\n--- [6/7] โหลดข้อมูลราคาน้ำมัน ---")
load_data_from_paths(files_oil)

print("\n--- โหลดดอกเบี้ยอ้างอิงแห่งประเทศไทย ---")
load_data_from_paths(files_th_interest)

print("\n--- [7/7] โหลดข้อมูลดัชนีตลาดโลก ---")
load_data_from_paths(files_indices)


=== เริ่มต้นการโหลดข้อมูลทั้งหมดเข้าสู่ระบบ ===

--- [1/7] โหลดข้อมูลหุ้นนำเข้า ---
✓ โหลด df_PTTGC สำเร็จ (2621 แถว x 7 คอลัมน์)
✓ โหลด df_TOP สำเร็จ (2621 แถว x 7 คอลัมน์)
✓ โหลด df_IRPC สำเร็จ (2621 แถว x 7 คอลัมน์)
✓ โหลด df_BGRIM สำเร็จ (2243 แถว x 7 คอลัมน์)
✓ โหลด df_EGCO สำเร็จ (2621 แถว x 7 คอลัมน์)
✓ โหลด df_BCP สำเร็จ (2761 แถว x 7 คอลัมน์)
✓ โหลด df_RATCH สำเร็จ (2621 แถว x 7 คอลัมน์)
✓ โหลด df_IVL สำเร็จ (2621 แถว x 7 คอลัมน์)
✓ โหลด df_COM7 สำเร็จ (2621 แถว x 7 คอลัมน์)
✓ โหลด df_GPSC สำเร็จ (2621 แถว x 7 คอลัมน์)

--- [2/7] โหลดข้อมูลหุ้นส่งออก ---
✓ โหลด df_KCE สำเร็จ (2621 แถว x 7 คอลัมน์)
✓ โหลด df_HANA สำเร็จ (2621 แถว x 7 คอลัมน์)
✓ โหลด df_CPF สำเร็จ (2621 แถว x 7 คอลัมน์)
✓ โหลด df_STA สำเร็จ (2621 แถว x 7 คอลัมน์)
✓ โหลด df_TU สำเร็จ (2621 แถว x 7 คอลัมน์)
✓ โหลด df_DELTA สำเร็จ (2621 แถว x 7 คอลัมน์)
✓ โหลด df_BH สำเร็จ (2621 แถว x 7 คอลัมน์)
✓ โหลด df_ITC สำเร็จ (933 แถว x 7 คอลัมน์)
✓ โหลด df_BDMS สำเร็จ (2621 แถว x 7 คอลัมน์)
✓ โหลด df_AAI สำเร็จ (960 แถว x 7

{'df_BDI':       วันเดือนปี    ล่าสุด  ราคาเปิด    สูงสุด    ต่ำสุด  ปริมาณ % เปลี่ยน
 0     11/18/2026  2,914.00  2,914.00  2,914.00  2,914.00     NaN     0.73%
 1     11/17/2026  2,893.00  2,893.00  2,893.00  2,893.00     NaN    -0.96%
 2     11/16/2026  2,921.00  2,921.00  2,921.00  2,921.00     NaN    -1.98%
 3     11/13/2026  2,980.00  2,980.00  2,980.00  2,980.00     NaN    -1.32%
 4     11/12/2026  3,020.00  3,020.00  3,020.00  3,020.00     NaN     2.06%
 ...          ...       ...       ...       ...       ...     ...       ...
 2700  01/08/2016    429.00    429.00    429.00    429.00     NaN    -3.60%
 2701  01/07/2016    445.00    445.00    445.00    445.00     NaN    -4.71%
 2702  01/06/2016    467.00    467.00    467.00    467.00     NaN    -0.21%
 2703  01/05/2016    468.00    468.00    468.00    468.00     NaN    -1.06%
 2704  01/04/2016    473.00    473.00    473.00    473.00     NaN    -1.05%
 
 [2705 rows x 7 columns],
 'df_DXY':       วันเดือนปี  ล่าสุด  ราคาเปิด  สูง

In [12]:
# สร้างลิสต์เก็บตัวแปร DataFrame ของกลุ่มหุ้นส่งออกโดยตรงเพื่อให้เรียกใช้ได้ง่าย
export_stocks_list = [
    df_DELTA, df_HANA, df_KCE, df_CPF, df_TU,
    df_AAI, df_STA, df_BDMS, df_BH, df_ITC
]

# สร้างลิสต์เก็บตัวแปร DataFrame ของกลุ่มหุ้นนำเข้าโดยตรง
import_stocks_list = [
    df_PTTGC, df_TOP, df_IRPC, df_IVL, df_COM7,
    df_BGRIM, df_GPSC, df_EGCO, df_RATCH, df_BCP
]

print(f"✓ สร้างลิสต์สำเร็จ: export_stocks_list มีหุ้นทั้งหมด {len(export_stocks_list)} ตัว")
print(f"✓ สร้างลิสต์สำเร็จ: import_stocks_list มีหุ้นทั้งหมด {len(import_stocks_list)} ตัว")

✓ สร้างลิสต์สำเร็จ: export_stocks_list มีหุ้นทั้งหมด 10 ตัว
✓ สร้างลิสต์สำเร็จ: import_stocks_list มีหุ้นทั้งหมด 10 ตัว


In [13]:
import pandas as pd

# ค้นหา DataFrame ทั้งหมดในระบบ
all_dfs = {name: val for name, val in globals().items() if isinstance(val, pd.DataFrame) and not name.startswith('_')}

print(f"พบ DataFrame ทั้งหมด {len(all_dfs)} ตัวแปรในระบบ:\n")
for name, df in sorted(all_dfs.items()):
    print(f"• {name} (ขนาด: {df.shape[0]} แถว x {df.shape[1]} คอลัมน์)")
    print(f"  คอลัมน์: {list(df.columns)}\n")

พบ DataFrame ทั้งหมด 37 ตัวแปรในระบบ:

• df_AAI (ขนาด: 960 แถว x 7 คอลัมน์)
  คอลัมน์: ['วันเดือนปี', 'ล่าสุด', 'ราคาเปิด', 'สูงสุด', 'ต่ำสุด', 'ปริมาณ', '% เปลี่ยน']

• df_BCP (ขนาด: 2761 แถว x 7 คอลัมน์)
  คอลัมน์: ['วันเดือนปี', 'ล่าสุด', 'ราคาเปิด', 'สูงสุด', 'ต่ำสุด', 'ปริมาณ', '% เปลี่ยน']

• df_BDI (ขนาด: 2705 แถว x 7 คอลัมน์)
  คอลัมน์: ['วันเดือนปี', 'ล่าสุด', 'ราคาเปิด', 'สูงสุด', 'ต่ำสุด', 'ปริมาณ', '% เปลี่ยน']

• df_BDMS (ขนาด: 2621 แถว x 7 คอลัมน์)
  คอลัมน์: ['วันเดือนปี', 'ล่าสุด', 'ราคาเปิด', 'สูงสุด', 'ต่ำสุด', 'ปริมาณ', '% เปลี่ยน']

• df_BGRIM (ขนาด: 2243 แถว x 7 คอลัมน์)
  คอลัมน์: ['วันเดือนปี', 'ล่าสุด', 'ราคาเปิด', 'สูงสุด', 'ต่ำสุด', 'ปริมาณ', '% เปลี่ยน']

• df_BH (ขนาด: 2621 แถว x 7 คอลัมน์)
  คอลัมน์: ['วันเดือนปี', 'ล่าสุด', 'ราคาเปิด', 'สูงสุด', 'ต่ำสุด', 'ปริมาณ', '% เปลี่ยน']

• df_CNY_THB (ขนาด: 2841 แถว x 7 คอลัมน์)
  คอลัมน์: ['วันเดือนปี', 'ล่าสุด', 'ราคาเปิด', 'สูงสุด', 'ต่ำสุด', 'ปริมาณ', '% เปลี่ยน']

• df_COM7 (ขนาด: 2621 แถว x 7 คอลัมน์)
  คอลัม

In [14]:
import pandas as pd

# กำหนดแมปปิ้งคำแปลสำหรับคอลัมน์ภาษาไทยเป็นภาษาอังกฤษตามที่กำหนด
column_mapping = {
    'วันเดือนปี': 'Date',
    'ล่าสุด': 'Close',
    'ราคาเปิด': 'Open',
    'สูงสุด': 'High',
    'ต่ำสุด': 'Low',
    'ปริมาณ': 'Volume',
    '% เปลี่ยน': 'Pct_Change',
    'ประเภทหุ้น': 'Asset_Type'  # เก็บคอลัมน์พิเศษเพิ่มเติมไว้ถ้ามี
}

# ดึงรายชื่อ DataFrame ทั้งหมดในระบบ
all_dfs = {name: val for name, val in globals().items() if isinstance(val, pd.DataFrame) and not name.startswith('_')}

print("=== เริ่มต้นทำการเปลี่ยนหัวคอลัมน์ให้เป็นรูปแบบมาตรฐาน ===\n")

for name, df in all_dfs.items():
    # ทำการแปลงชื่อคอลัมน์ตามแมปปิ้งที่มี
    df.rename(columns=column_mapping, inplace=True)

    # หากขาดคอลัมน์ 'Volume' (เช่นในกรณีของข้อมูลพันธบัตร) จะทำการเพิ่มคอลัมน์ 'Volume' และเติมค่า NaN หรือ '-' เพื่อให้โครงสร้างเหมือนกัน
    if 'Volume' not in df.columns:
        df['Volume'] = None

    # จัดเรียงลำดับคอลัมน์พื้นฐานให้อยู่ลำดับแรกเพื่อความเป็นระเบียบ
    base_cols = ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
    existing_base = [col for col in base_cols if col in df.columns]
    extra_cols = [col for col in df.columns if col not in base_cols]

    # กำหนดลำดับคอลัมน์ใหม่
    df = df[existing_base + extra_cols]
    globals()[name] = df # อัปเดตกลับไปยังตัวแปร Global

    print(f"✓ {name} ได้รับการปรับปรุงหัวคอลัมน์แล้ว -> {list(df.columns)}")

=== เริ่มต้นทำการเปลี่ยนหัวคอลัมน์ให้เป็นรูปแบบมาตรฐาน ===

✓ df_PTTGC ได้รับการปรับปรุงหัวคอลัมน์แล้ว -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_TOP ได้รับการปรับปรุงหัวคอลัมน์แล้ว -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_IRPC ได้รับการปรับปรุงหัวคอลัมน์แล้ว -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_BGRIM ได้รับการปรับปรุงหัวคอลัมน์แล้ว -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_EGCO ได้รับการปรับปรุงหัวคอลัมน์แล้ว -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_BCP ได้รับการปรับปรุงหัวคอลัมน์แล้ว -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_RATCH ได้รับการปรับปรุงหัวคอลัมน์แล้ว -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_IVL ได้รับการปรับปรุงหัวคอลัมน์แล้ว -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_COM7 ได้รับการปรับปรุงหัวคอลัมน์แล้ว -> ['Date', 'Close

In [15]:
import pandas as pd

# ดึงรายชื่อ DataFrame ทั้งหมดในระบบ
all_dfs = {name: val for name, val in globals().items() if isinstance(val, pd.DataFrame) and not name.startswith('_')}

target_cols = ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']

print("=== เริ่มต้นการลบคอลัมน์ที่ไม่เกี่ยวข้องออก ===\n")

for name, df in all_dfs.items():
    # เลือกเฉพาะคอลัมน์ที่อยู่ในเป้าหมายและมีอยู่ใน DataFrame จริง
    existing_target_cols = [col for col in target_cols if col in df.columns]

    # อัปเดต DataFrame โดยเก็บไว้เฉพาะ 7 คอลัมน์หลัก
    df_filtered = df[existing_target_cols]
    globals()[name] = df_filtered

    print(f"✓ {name} คงเหลือเฉพาะคอลัมน์ -> {list(df_filtered.columns)}")

=== เริ่มต้นการลบคอลัมน์ที่ไม่เกี่ยวข้องออก ===

✓ df_PTTGC คงเหลือเฉพาะคอลัมน์ -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_TOP คงเหลือเฉพาะคอลัมน์ -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_IRPC คงเหลือเฉพาะคอลัมน์ -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_BGRIM คงเหลือเฉพาะคอลัมน์ -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_EGCO คงเหลือเฉพาะคอลัมน์ -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_BCP คงเหลือเฉพาะคอลัมน์ -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_RATCH คงเหลือเฉพาะคอลัมน์ -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_IVL คงเหลือเฉพาะคอลัมน์ -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_COM7 คงเหลือเฉพาะคอลัมน์ -> ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Pct_Change']
✓ df_GPSC คงเหลือเฉพาะคอลัมน์ -> ['Date', 'Close', 'Open', 'High', 'L

# CLEANING DATA

## Null

### Null checking

In [16]:
# ค้นหาและเก็บทุก DataFrame ที่อยู่ในระบบมาวนลูปตรวจสอบค่าว่าง
all_dfs = {name: val for name, val in globals().items() if isinstance(val, pd.DataFrame) and not name.startswith('_')}

print(f"--- กำลังเริ่มตรวจสอบค่าว่างในทั้งหมด {len(all_dfs)} DataFrames ---\n")

for name, df in sorted(all_dfs.items()):
    # ข้ามตัวแปรพักข้อมูลชั่วคราวที่เป็น df เพื่อไม่ให้ซ้ำซ้อน
    if name == 'df':
        continue

    # ตรวจสอบเบื้องต้นว่ามีค่าว่างใน DataFrame นี้หรือไม่
    null_sum = df.isnull().sum().sum()
    if null_sum > 0:
        print("=" * 60)
        print(f"📌 [พบค่าว่างใน DataFrame: {name}] (รวมทั้งหมด {null_sum} ตำแหน่ง)")
        print("=" * 60)
        check_and_show_nulls(df)
        print("\n")
    else:
        print(f"✓ {name}: ไม่มีค่าว่าง")

--- กำลังเริ่มตรวจสอบค่าว่างในทั้งหมด 39 DataFrames ---

✓ df_AAI: ไม่มีค่าว่าง
📌 [พบค่าว่างใน DataFrame: df_BCP] (รวมทั้งหมด 1 ตำแหน่ง)
⚠ พบค่าว่างในคอลัมน์:
  - 'Volume': 1 แถว

รวมแถวที่มีค่าว่างทั้งหมด: 1 แถว


,Date,Close,Open,High,Low,Volume,Pct_Change
530,09/16/2024,0.4078,0.4128,0.4131,0.4038,NaN,-1.26%




📌 [พบค่าว่างใน DataFrame: df_BDI] (รวมทั้งหมด 2705 ตำแหน่ง)
⚠ พบค่าว่างในคอลัมน์:
  - 'Volume': 2705 แถว

รวมแถวที่มีค่าว่างทั้งหมด: 2705 แถว


,Date,Close,Open,High,Low,Volume,Pct_Change
0,11/18/2026,"2,914.00","2,914.00","2,914.00","2,914.00",NaN,0.73%
1,11/17/2026,"2,893.00","2,893.00","2,893.00","2,893.00",NaN,-0.96%
2,11/16/2026,"2,921.00","2,921.00","2,921.00","2,921.00",NaN,-1.98%
3,11/13/2026,"2,980.00","2,980.00","2,980.00","2,980.00",NaN,-1.32%
4,11/12/2026,"3,020.00","3,020.00","3,020.00","3,020.00",NaN,2.06%
5,11/11/2026,"2,959.00","2,959.00","2,959.00","2,959.00",NaN,3.39%
6,11/10/2026,"2,862.00","2,862.00","2,862.00","2,862.00",NaN,0.28%
7,11/09/2026,"2,854.00","2,854.00","2,854.00","2,854.00",NaN,1.75%
8,11/06/2026,"2,805.00","2,805.00","2,805.00","2,805.00",NaN,2.37%
9,11/05/2026,"2,740.00","2,740.00","2,740.00","2,740.00",NaN,1.71%




✓ df_BDMS: ไม่มีค่าว่าง
✓ df_BGRIM: ไม่มีค่าว่าง
✓ df_BH: ไม่มีค่าว่าง
📌 [พบค่าว่างใน DataFrame: df_CNY_THB] (รวมทั้งหมด 2841 ตำแหน่ง)
⚠ พบค่าว่างในคอลัมน์:
  - 'Volume': 2841 แถว

รวมแถวที่มีค่าว่างทั้งหมด: 2841 แถว


,Date,Close,Open,High,Low,Volume,Pct_Change
0,11/20/2026,5.0819,5.0973,5.0976,5.0752,NaN,-0.27%
1,11/19/2026,5.0955,5.0947,5.1107,5.0766,NaN,0.02%
2,11/18/2026,5.0947,5.0696,5.0988,5.0696,NaN,0.50%
3,11/17/2026,5.0696,5.0859,5.0901,5.0659,NaN,-0.32%
4,11/16/2026,5.0859,5.0827,5.0901,5.0699,NaN,0.15%
5,11/13/2026,5.0782,5.0844,5.0877,5.0678,NaN,-0.12%
6,11/12/2026,5.0844,5.0609,5.0874,5.0549,NaN,0.46%
7,11/11/2026,5.0609,5.0410,5.0654,5.0270,NaN,0.39%
8,11/10/2026,5.0410,5.0099,5.0447,4.9962,NaN,0.65%
9,11/09/2026,5.0084,5.0281,5.0352,5.0009,NaN,-0.39%


... (24 outputs left out)



📌 [พบค่าว่างใน DataFrame: df_filtered] (รวมทั้งหมด 1 ตำแหน่ง)
⚠ พบค่าว่างในคอลัมน์:
  - 'Volume': 1 แถว

รวมแถวที่มีค่าว่างทั้งหมด: 1 แถว


,Date,Close,Open,High,Low,Volume,Pct_Change
2526,01/02/2017,56.82,56.82,56.82,56.82,NaN,0.00%




📌 [พบค่าว่างใน DataFrame: df_เบรนท์] (รวมทั้งหมด 1 ตำแหน่ง)
⚠ พบค่าว่างในคอลัมน์:
  - 'Volume': 1 แถว

รวมแถวที่มีค่าว่างทั้งหมด: 1 แถว


,Date,Close,Open,High,Low,Volume,Pct_Change
2526,01/02/2017,56.82,56.82,56.82,56.82,NaN,0.00%


### Null handling

#### ลบคอลลัมน์ที่เป็นnull ออก

In [17]:
# รายชื่อ DataFrame ที่ต้องการลบคอลัมน์ 'Volume' (สินทรัพย์ที่ไม่ใช้ Volume หรือเป็นดัชนี/อัตราแลกเปลี่ยน)
non_equity_vars = [
    'df_TH_10Y', 'df_TH_20Y', 'df_USD_THB', 'df_CNY_THB', 'df_EUR_THB', 'df_JPY_THB',
    'df_US_10Y', 'df_US_30Y', 'df_BDI', 'df_VIX', 'df_SOX', 'df_SP500', 'df_SET50', 'df_DXY', 'df_TH_RATE'
]

for name in non_equity_vars:
    df = globals().get(name)
    if df is not None and not df.empty and 'Volume' in df.columns:
        print(f"[{name}]")
        globals()[name] = drop_columns(df, ['Volume'])
        print()


[df_TH_10Y]
✓ ลบคอลัมน์สำเร็จ: ['Volume']

[df_TH_20Y]
✓ ลบคอลัมน์สำเร็จ: ['Volume']

[df_USD_THB]
✓ ลบคอลัมน์สำเร็จ: ['Volume']

[df_CNY_THB]
✓ ลบคอลัมน์สำเร็จ: ['Volume']

[df_EUR_THB]
✓ ลบคอลัมน์สำเร็จ: ['Volume']

[df_JPY_THB]
✓ ลบคอลัมน์สำเร็จ: ['Volume']

[df_US_10Y]
✓ ลบคอลัมน์สำเร็จ: ['Volume']

[df_US_30Y]
✓ ลบคอลัมน์สำเร็จ: ['Volume']

[df_BDI]
✓ ลบคอลัมน์สำเร็จ: ['Volume']

[df_VIX]
✓ ลบคอลัมน์สำเร็จ: ['Volume']

[df_SOX]
✓ ลบคอลัมน์สำเร็จ: ['Volume']

[df_SP500]
✓ ลบคอลัมน์สำเร็จ: ['Volume']

[df_SET50]
✓ ลบคอลัมน์สำเร็จ: ['Volume']

[df_DXY]
✓ ลบคอลัมน์สำเร็จ: ['Volume']

[df_TH_RATE]
✓ ลบคอลัมน์สำเร็จ: ['Volume']



#### Bfill and Ffill null rows

In [18]:
# ดึงรายชื่อ DataFrame ทั้งหมดในระบบ
all_dfs = {name: val for name, val in globals().items() if isinstance(val, pd.DataFrame) and not name.startswith('_')}

print("=== เริ่มต้นจัดการลบแถวที่มีค่าว่าง (Null Rows) ===\n")

for name, df in sorted(all_dfs.items()):
    if name == 'df':
        continue

    null_rows_count = df.isnull().any(axis=1).sum()
    if null_rows_count > 0:
        print(f"[{name}] พบแถวที่มีค่าว่างทั้งหมด {null_rows_count} แถว")
        globals()[name] = drop_null_rows(df)
        print()

=== เริ่มต้นจัดการลบแถวที่มีค่าว่าง (Null Rows) ===

[df_BCP] พบแถวที่มีค่าว่างทั้งหมด 1 แถว
✓ ลบค่าว่างสำเร็จ: ทั้งหมด 1 แถว

[df_WTI] พบแถวที่มีค่าว่างทั้งหมด 111 แถว
✓ ลบค่าว่างสำเร็จ: ทั้งหมด 111 แถว

[df_filtered] พบแถวที่มีค่าว่างทั้งหมด 1 แถว
✓ ลบค่าว่างสำเร็จ: ทั้งหมด 1 แถว

[df_เบรนท์] พบแถวที่มีค่าว่างทั้งหมด 1 แถว
✓ ลบค่าว่างสำเร็จ: ทั้งหมด 1 แถว



### Null checking

In [19]:
# ค้นหาและเก็บทุก DataFrame ที่อยู่ในระบบมาวนลูปตรวจสอบค่าว่าง
all_dfs = {name: val for name, val in globals().items() if isinstance(val, pd.DataFrame) and not name.startswith('_')}

print(f"--- กำลังเริ่มตรวจสอบค่าว่างในทั้งหมด {len(all_dfs)} DataFrames ---\n")

for name, df in sorted(all_dfs.items()):
    # ข้ามตัวแปรพักข้อมูลชั่วคราวที่เป็น df เพื่อไม่ให้ซ้ำซ้อน
    if name == 'df':
        continue
    check_and_show_nulls(df)

--- กำลังเริ่มตรวจสอบค่าว่างในทั้งหมด 39 DataFrames ---

✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบค่าว่างในข้อมูลนี้
✓ ไม่พบ

# บันทึกข้อมูล Intermediate (Cleaned Data)
ส่งออกชุดข้อมูลทั้งหมดที่ทำความสะอาดแล้วไปยังไดเรกทอรี `data/02_intermediate/` เพื่อให้ Notebook ขั้นตอนต่อไปสามารถโหลดใช้งานได้ทันทีโดยไม่ต้องรันขั้นตอนทำความสะอาดซ้ำ

In [20]:
import os
from pathlib import Path
import pandas as pd

# กำหนด Path บันทึกข้อมูล Cleaned ไปยัง data/02_intermediate/
save_dir = Path('../data/02_intermediate')
if not save_dir.parent.exists():
    save_dir = Path('data/02_intermediate')
save_dir.mkdir(parents=True, exist_ok=True)

saved_count = 0
all_dfs = {name: val for name, val in globals().items() if isinstance(val, pd.DataFrame) and name.startswith('df_') and not name.startswith('df_temp')}

for name, df in sorted(all_dfs.items()):
    out_file = save_dir / f"{name}.csv"
    df.to_csv(out_file, index=False)
    saved_count += 1

print(f"✓ บันทึกข้อมูลที่คลีนเรียบร้อยแล้วทั้งหมด {saved_count} ไฟล์ ลงใน: {save_dir.resolve()}")


✓ บันทึกข้อมูลที่คลีนเรียบร้อยแล้วทั้งหมด 38 ไฟล์ ลงใน: /Users/pathcharapisit/Desktop/newdsde/data/02_intermediate
